In [1]:
# CELL 1: Environment Setup & Cloud Run Memory Purge
import os
import sys
import site
import importlib

# 1. Install packages
!pip install "numpy<2.0" "scipy<1.13" bitsandbytes --force-reinstall --no-cache-dir
!pip install torch==2.5.1 torchvision==0.20.1 transformers==4.46.2 datasets==3.1.0 tqdm==4.67.0 accelerate --no-cache-dir

# 2. Force Python to rescan the site-packages directory on the disk
importlib.reload(site)

# 3. Purge pre-loaded modules from active RAM
# This tricks Python into thinking these libraries were never imported, 
# forcing it to load the freshly installed versions in the next step.
modules_to_purge = ['numpy', 'scipy', 'torch', 'transformers', 'datasets', 'accelerate', 'bitsandbytes']
for module_name in list(sys.modules.keys()):
    if any(module_name.startswith(m) for m in modules_to_purge):
        del sys.modules[module_name]

# 4. Perform the actual imports
import json
import time
import random
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
import numpy as np
import bitsandbytes as bnb
from collections import namedtuple
from transformers import AutoModelForCausalLM, AutoTokenizer, PreTrainedTokenizerBase, DynamicCache
from transformers.data.data_collator import pad_without_fast_tokenizer_warning
from datasets import load_dataset, Dataset as HFDataset
from torch.utils.data import DataLoader
from tqdm.notebook import tqdm

# Disable tokenizer parallelism to prevent deadlocks
os.environ["TOKENIZERS_PARALLELISM"] = "false"
print("Environment setup complete. Fresh modules loaded successfully into RAM.")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 6.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.4/60.4 kB 138.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.0/18.0 MB 279.2 MB/s eta 0:00:00 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 37.8/37.8 MB 292.2 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 283.5 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 74.4/74.4 kB 288.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 915.7/915.7 MB 314.2 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.2/12.2 MB 265.4 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 594.3/594.3 MB 277.1 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 260.0 MB/s eta 0:00:00 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 88.0/88.0 MB 281.2 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 95

2026-03-05 08:40:08.055262: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1772700008.276799      55 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1772700008.331893      55 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1772700008.815941      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1772700008.815982      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1772700008.815985      55 computation_placer.cc:177] computation placer alr

Environment setup complete. Fresh modules loaded successfully into RAM.


In [2]:
# CELL 2: Centralized Configuration (Qwen2.5 Engine + Stabilized)
class Config:
    def __init__(self):
        self.model_id = "Qwen/Qwen2.5-1.5B-Instruct" 
        self.save_path = "./checkpoints_coconut_steered"
        
        self.train_split_1_ratio = 0.60
        self.train_split_2_ratio = 0.10
        self.train_split_3_ratio = 0.30
        
        self.batch_size_training = 1       
        self.gradient_accumulation_steps = 128  
        
        # --- STABILITY FIX: Lowered LR for 1.5B Model ---
        self.lr = 2e-5
        
        self.weight_decay = 0.01
        
        # --- MEMORY OPTIMIZATION: Trimmed to 512 ---
        self.max_seq_len = 512             
        
        self.num_epochs_phase1 = 18 
        self.c_thought = 2          
        self.max_latent_stage = 3   
        
        self.num_generations_per_sample = 1 
        self.generation_temperature = 0.0   
        
        self.alpha_sweep = [0.0, 5.0, 10.0, 20.0, 50.0]
        self.alpha_decay = 0.7 
        
        self.seed = 42
        self.bf16 = True if torch.cuda.is_available() and torch.cuda.is_bf16_supported() else False
        self.device = "cuda" if torch.cuda.is_available() else "cpu"

configs = Config()

def set_seed(seed_value):
    random.seed(seed_value)
    np.random.seed(seed_value)
    torch.manual_seed(seed_value)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed_value)

set_seed(configs.seed)

In [3]:
# CELL 3: Dataset Preparation & Reduced 60/10/30 Splitting
def parse_example(ex):
    raw_answer = ex["answer"]
    if "####" in raw_answer:
        reasoning, final_ans = raw_answer.split("####")
        final_ans = final_ans.strip()
    else:
        reasoning = raw_answer
        final_ans = ""
    steps = [s.strip() for s in reasoning.split("\n") if s.strip()]
    return {"question": ex["question"], "steps": steps, "answer": final_ans}

print("Loading and splitting GSM8K dataset...")
raw_dataset = load_dataset("gsm8k", "main")

# --- REDUCED DATASET SIZE ---
# Change this number to scale your experiment up or down
MAX_SAMPLES = 2000 
train_full = [parse_example(ex) for ex in raw_dataset["train"]][:MAX_SAMPLES]

# --- DYNAMIC CALCULATION: 60% / 10% / 30% ---
total_samples = len(train_full)
idx_phase1_end = int(total_samples * configs.train_split_1_ratio)                 
idx_phase2_end = idx_phase1_end + int(total_samples * configs.train_split_2_ratio) 

# Apply strict isolation based on calculated indices
data_phase1 = train_full[:idx_phase1_end]
data_phase2 = train_full[idx_phase1_end : idx_phase2_end]
data_phase3 = train_full[idx_phase2_end :] 

# Official Test Dataset
test_data = [parse_example(ex) for ex in raw_dataset["test"]]

print(f"Total Train Sub-Dataset Used: {total_samples}")
print(f"Phase 1 (Base Train - 60%): {len(data_phase1)}")
print(f"Phase 2 (Vector Extraction - 10%): {len(data_phase2)}")
print(f"Phase 3 (Validation - 30%): {len(data_phase3)}")
print(f"Phase 4 (Full Test Set): {len(test_data)}")

def get_hf_dataset(raw_data, tokenizer):
    def tokenize_sample(sample):
        q_tok = tokenizer.encode(sample["question"] + "\n", add_special_tokens=True)
        s_tok = [tokenizer.encode(s + "\n", add_special_tokens=False) for s in sample["steps"]]
        a_tok = tokenizer.encode("#### " + sample["answer"], add_special_tokens=False) + [tokenizer.eos_token_id]
        return {"question_tokenized": q_tok, "steps_tokenized": s_tok, "answer_tokenized": a_tok, "ground_truth": sample["answer"]}
    
    ds = HFDataset.from_list(raw_data)
    return ds.map(tokenize_sample, num_proc=4)

Loading and splitting GSM8K dataset...


README.md: 0.00B [00:00, ?B/s]

main/train-00000-of-00001.parquet:   0%|          | 0.00/2.31M [00:00<?, ?B/s]

main/test-00000-of-00001.parquet:   0%|          | 0.00/419k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/7473 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/1319 [00:00<?, ? examples/s]

Total Train Sub-Dataset Used: 2000
Phase 1 (Base Train - 60%): 1200
Phase 2 (Vector Extraction - 10%): 200
Phase 3 (Validation - 30%): 600
Phase 4 (Full Test Set): 1319


In [4]:
# CELL 4: COCONUT Architecture with Dynamic Alpha Decay ITI
import itertools
import numpy as np
from dataclasses import dataclass
from torch.nn import CrossEntropyLoss
import torch.nn.functional as F

Outputs = namedtuple("Outputs", ["loss", "inputs_embeds", "logits", "latent_sequence"])

class Coconut(nn.Module):
    def __init__(self, base_causallm, latent_token_id, start_latent_id, end_latent_id, eos_token_id):
        super().__init__()
        self.base_causallm = base_causallm
        self.latent_token_id = latent_token_id
        self.eos_token_id = eos_token_id
        self.embedding = self.base_causallm.get_input_embeddings()

    def _process_kv(self, kv_cache, keep_len):
        if kv_cache is None: return None
        new_cache = DynamicCache()
        num_layers = len(kv_cache.key_cache) if hasattr(kv_cache, "key_cache") else len(kv_cache)
        for i in range(num_layers):
            try:
                if hasattr(kv_cache, "key_cache"):
                    k, v = kv_cache.key_cache[i], kv_cache.value_cache[i]
                else:
                    k, v = kv_cache[i]
            except:
                k, v = kv_cache[i]
            new_cache.update(k[..., :keep_len, :], v[..., :keep_len, :], layer_idx=i)
        return new_cache

    def forward(self, input_ids, attention_mask, labels=None, position_ids=None, steering_vector=None, alpha=0.0, gamma=1.0):
        logits = []
        latent_sequence = [] 
        
        latent_indices = (input_ids == self.latent_token_id).nonzero()
        latent_lists = [[idx[1].item() for idx in latent_indices if idx[0] == i] for i in range(input_ids.shape[0])]
        max_n_latents = max([len(l) for l in latent_lists]) if latent_lists else 0

        inputs_embeds = self.embedding(input_ids)
        next_compute_range = (0, input_ids.shape[1])
        if max_n_latents > 0:
            next_compute_range = (0, latent_indices[:, 1].min().item())

        kv_cache = None

        for pass_idx in range(max_n_latents):
            curr_cache = self._process_kv(kv_cache, next_compute_range[0])
            outputs = self.base_causallm(
                inputs_embeds=inputs_embeds[:, next_compute_range[0] : next_compute_range[1]],
                attention_mask=attention_mask[:, :next_compute_range[1]],
                past_key_values=curr_cache,
                output_hidden_states=True,
                use_cache=False if self.training else True
            )
            logits.append(outputs.logits)

            next_end = input_ids.shape[1] if pass_idx + 1 >= max_n_latents else next_compute_range[1] + 1
            next_compute_range = (next_compute_range[1], next_end)

            hidden_states = outputs.hidden_states[-1]
            
            # --- MODIFIED: Dynamic Alpha Decay ---
            if steering_vector is not None and alpha > 0:
                current_alpha = alpha * (gamma ** pass_idx) # Decay alpha based on the depth of the latent step
                sigma_l = hidden_states.std(dim=-1, keepdim=True)
                norm_dir = F.normalize(steering_vector, p=2, dim=-1).to(hidden_states.device)
                hidden_states = hidden_states + (current_alpha * sigma_l * norm_dir)

            latent_sequence.append(hidden_states.detach())
            
            kv_cache = outputs.past_key_values
            inputs_embeds = inputs_embeds.clone()

            filling_indices = [(i, l[pass_idx]) for i, l in enumerate(latent_lists) if len(l) > pass_idx]
            for batch_idx, token_idx in filling_indices:
                inputs_embeds[batch_idx, token_idx] = hidden_states[batch_idx, -1]

        final_cache = self._process_kv(kv_cache, next_compute_range[0])
        outputs = self.base_causallm(
            inputs_embeds=inputs_embeds[:, next_compute_range[0] : next_compute_range[1]],
            attention_mask=attention_mask[:, :next_compute_range[1]],
            past_key_values=final_cache,
            use_cache=False if self.training else True
        )
        logits.append(outputs.logits)
        logits = torch.cat(logits, dim=-2)
        
        loss = None
        if labels is not None:
            shift_logits = logits[..., :-1, :].contiguous()
            shift_labels = labels[..., 1:].contiguous()
            loss = CrossEntropyLoss()(shift_logits.view(-1, shift_logits.size(-1)), shift_labels.view(-1))

        return Outputs(loss, inputs_embeds, logits, latent_sequence)

    def generate_with_latents(self, input_ids, max_new_tokens=128, temperature=0.0, steering_vector=None, alpha=0.0, gamma=1.0):
        self.eval()
        tokens = input_ids.tolist()[0]

        with torch.no_grad():
            outputs = self.forward(input_ids=input_ids, attention_mask=torch.ones_like(input_ids), steering_vector=steering_vector, alpha=alpha, gamma=gamma)
            
        mean_latent = torch.mean(torch.stack([h[:, -1, :] for h in outputs.latent_sequence]), dim=0).cpu() if outputs.latent_sequence else None
        
        faithfulness_scores = []
        if steering_vector is not None and outputs.latent_sequence:
            for h in outputs.latent_sequence:
                sim = F.cosine_similarity(h[:, -1, :], steering_vector.unsqueeze(0), dim=-1)
                faithfulness_scores.append(sim.item())
        avg_faithfulness = np.mean(faithfulness_scores) if faithfulness_scores else 0.0

        if temperature > 0:
            scaled_logits = outputs.logits[:, -1, :] / temperature
            probs = F.softmax(scaled_logits, dim=-1)
            next_token = torch.multinomial(probs, num_samples=1).item()
        else:
            next_token = torch.argmax(outputs.logits[:, -1, :], dim=-1).item()
            
        tokens.append(next_token)
        curr_input_ids = torch.tensor([tokens], device=input_ids.device)
        
        for _ in range(max_new_tokens):
            with torch.no_grad():
                outputs = self.base_causallm(input_ids=curr_input_ids)
                
            if temperature > 0:
                scaled_logits = outputs.logits[:, -1, :] / temperature
                probs = F.softmax(scaled_logits, dim=-1)
                next_token = torch.multinomial(probs, num_samples=1).item()
            else:
                next_token = torch.argmax(outputs.logits[:, -1, :], dim=-1).item()
                
            if next_token == self.eos_token_id: break
            tokens.append(next_token)
            curr_input_ids = torch.tensor([tokens], device=input_ids.device)

        return torch.tensor([tokens]), mean_latent, avg_faithfulness

In [ ]:
from dataclasses import dataclass
from torch.nn import CrossEntropyLoss


In [ ]:
# CELL 5: Phase 1 - Base COCONUT Training (Paged Optimizer + Gradient Clipping)
import gc
from dataclasses import dataclass
import itertools
import bitsandbytes as bnb
import torch.nn as nn

def get_stage_info(epoch):
    if epoch < 6: return 0, False, (epoch == 0)   
    elif epoch < 9: return 1, False, (epoch == 6)   
    elif epoch < 12: return 2, False, (epoch == 9)   
    elif epoch < 15: return 3, False, (epoch == 12)  
    else: return 3, True, (epoch == 15)   

def get_cot_latent_dataset(scheduled_stage, drop_remaining, base_dataset, configs, start_id, latent_id, end_id, shuffle=False):
    def process_dataset(sample):
        n_steps_total = len(sample["steps_tokenized"])
        n_steps_to_latentize = min(scheduled_stage, n_steps_total)
        if n_steps_to_latentize > configs.max_latent_stage:
             n_steps_to_latentize = configs.max_latent_stage
        
        n_latent_tokens = n_steps_to_latentize * configs.c_thought
        
        if drop_remaining: remaining_steps = []
        else: remaining_steps = list(itertools.chain.from_iterable(sample["steps_tokenized"][n_steps_to_latentize:]))
            
        tokens = (sample["question_tokenized"] + [start_id] + [latent_id] * n_latent_tokens + [end_id] +
                  remaining_steps + sample["answer_tokenized"])
        
        mask_len = len(sample["question_tokenized"]) + n_latent_tokens + 2
        labels = [-100] * mask_len + tokens[mask_len:]
        
        tokens = tokens[:configs.max_seq_len]
        labels = labels[:configs.max_seq_len]
        
        return {"input_ids": tokens, "labels": labels, "attention_mask": [1] * len(tokens)}
    
    dataset = base_dataset.map(process_dataset, remove_columns=list(base_dataset.features))
    if shuffle: dataset = dataset.shuffle(seed=configs.seed)
    return dataset

@dataclass
class MyCollator:
    tokenizer: PreTrainedTokenizerBase
    latent_id: int
    label_pad_token_id: int = -100
    def __call__(self, features):
        earliest_latent = [f["input_ids"].index(self.latent_id) for f in features if self.latent_id in f["input_ids"]]
        if earliest_latent:
            latest_earliest = max(earliest_latent)
            for feature in features:
                pad = latest_earliest - feature["input_ids"].index(self.latent_id) if self.latent_id in feature["input_ids"] else 0
                feature["input_ids"] = [self.tokenizer.pad_token_id] * pad + feature["input_ids"]
                feature["attention_mask"] = [0] * pad + feature["attention_mask"]
                if "labels" in feature: feature["labels"] = [self.label_pad_token_id] * pad + feature["labels"]
        
        labels = [f.pop("labels") for f in features] if "labels" in features[0] else None
        batch = pad_without_fast_tokenizer_warning(self.tokenizer, features, padding=True, return_tensors="pt")
        if labels:
             max_len = batch["input_ids"].shape[1]
             batch["labels"] = torch.tensor([l + [self.label_pad_token_id]*(max_len-len(l)) for l in labels])
        return batch

print("Initializing Qwen Model for Phase 1...")
dt = torch.bfloat16 if configs.bf16 else torch.float32
model = AutoModelForCausalLM.from_pretrained(configs.model_id, torch_dtype=dt, trust_remote_code=True)
model.gradient_checkpointing_enable()

tokenizer = AutoTokenizer.from_pretrained(configs.model_id, trust_remote_code=True)
if tokenizer.pad_token is None: tokenizer.pad_token = tokenizer.eos_token

tokenizer.add_tokens(["<|start-latent|>", "<|end-latent|>", "<|latent|>"])
latent_id, start_id, end_id = tokenizer.convert_tokens_to_ids(["<|latent|>", "<|start-latent|>", "<|end-latent|>"])
model.resize_token_embeddings(len(tokenizer))

with torch.no_grad():
    input_embeds = model.get_input_embeddings()
    init_id = tokenizer.encode("The", add_special_tokens=False)[0] 
    input_embeds.weight.data[latent_id] = input_embeds.weight.data[init_id].clone()
    if hasattr(model, 'lm_head') and model.lm_head is not None:
        model.lm_head.weight.data[latent_id] = model.lm_head.weight.data[init_id].clone()
    input_embeds.weight.requires_grad = True

coconut_model = Coconut(model, latent_id, start_id, end_id, tokenizer.eos_token_id).to(configs.device)
collator = MyCollator(tokenizer, latent_id=latent_id)
ds_phase1 = get_hf_dataset(data_phase1, tokenizer)

optimizer = None
loss_history = [] 

print("Starting Phase 1 Base COCONUT Training (Stabilized)...")
for epoch in range(configs.num_epochs_phase1):
    
    current_stage, drop_remaining, requires_reset = get_stage_info(epoch)
    
    if requires_reset or optimizer is None:
        print(f"\n[Epoch {epoch}] Stage shifted to {current_stage}. Hard resetting PagedAdamW8bit Optimizer...")
        # --- STABILITY FIX: Paged Optimizer to prevent OOM ---
        optimizer = bnb.optim.PagedAdamW8bit(coconut_model.parameters(), lr=configs.lr, weight_decay=configs.weight_decay)
        torch.cuda.empty_cache()

    train_ds = get_cot_latent_dataset(current_stage, drop_remaining, ds_phase1, configs, start_id, latent_id, end_id, shuffle=True)
    train_loader = DataLoader(train_ds, batch_size=configs.batch_size_training, collate_fn=collator, shuffle=True)

    coconut_model.train()
    total_loss = 0
    pbar = tqdm(train_loader, desc=f"Epoch {epoch+1}/18 | Stage {current_stage} | Drop Text: {drop_remaining}")
    
    for step, batch in enumerate(pbar):
        input_ids = batch["input_ids"].to(configs.device)
        attention_mask = batch["attention_mask"].to(configs.device)
        labels = batch["labels"].to(configs.device)

        outputs = coconut_model(input_ids, attention_mask, labels)
        
        loss_history.append(outputs.loss.item())

        loss = outputs.loss / configs.gradient_accumulation_steps
        loss.backward()

        if (step + 1) % configs.gradient_accumulation_steps == 0:
            # --- STABILITY FIX: Gradient Clipping ---
            nn.utils.clip_grad_norm_(coconut_model.parameters(), max_norm=1.0)
            
            optimizer.step()
            optimizer.zero_grad()
            torch.cuda.empty_cache()

        total_loss += loss.item() * configs.gradient_accumulation_steps
        pbar.set_postfix({"loss": total_loss / (step + 1)})

Initializing Qwen Model for Phase 1...


config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/3.09G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

...


[Epoch 12] Stage shifted to 3. Hard resetting PagedAdamW8bit Optimizer...


Map:   0%|          | 0/1200 [00:00<?, ? examples/s]

Epoch 13/18 | Stage 3 | Drop Text: False:   0%|          | 0/1200 [00:00<?, ?it/s]

Map:   0%|          | 0/1200 [00:00<?, ? examples/s]

Epoch 14/18 | Stage 3 | Drop Text: False:   0%|          | 0/1200 [00:00<?, ?it/s]

In [ ]:
# CELL 5.1: Phase 1 Qualitative Output Check
import gc
import torch   

def print_sample_outputs(model, tokenizer, dataset, num_samples=2, phase_name=""):
    print(f"\n{'='*20} {phase_name} OUTPUT CHECK {'='*20}")
    model.eval()
    n_latents = configs.max_latent_stage * configs.c_thought
    
    # Pick a few fixed samples for consistent comparison across phases
    sample_indices = [0, 1] if len(dataset) >= 2 else range(len(dataset))
    
    for idx in sample_indices:
        sample = dataset[idx]
        prompt = sample["question"] + "\n<|start-latent|>" + "<|latent|>" * n_latents + "<|end-latent|>" 
        input_ids = tokenizer.encode(prompt, return_tensors="pt").to(configs.device)
        
        with torch.no_grad():
            # --- BUG FIX: Unpacking 3 variables to account for Trajectory Faithfulness ---
            generated_ids, _, _ = model.generate_with_latents(
                input_ids, 
                max_new_tokens=64, 
                temperature=0.0 # Greedy decode for clean comparison
            )
            
        output_text = tokenizer.decode(generated_ids[0], skip_special_tokens=True)
        # Extract the part generated after our prompt
        prompt_decoded = tokenizer.decode(input_ids[0], skip_special_tokens=True)
        new_generation = output_text.replace(prompt_decoded, "").strip()
        
        print(f"\n[Question {idx+1}]: {sample['question']}")
        print(f"[Ground Truth]: {sample['answer']}")
        print(f"[Model Output]: {new_generation}")
        print("-" * 60)
        
    # Free up memory before the next training phase
    del input_ids, generated_ids
    torch.cuda.empty_cache()
    gc.collect()

# Run the check for Phase 1
print_sample_outputs(coconut_model, tokenizer, test_data, num_samples=2, phase_name="PHASE 1 (BASE)")

In [ ]:
# CELL 5.2: Phase 1 Training Loss Visualization
import matplotlib.pyplot as plt

# Assuming you tracked 'loss_history' during the Phase 1 loop
if 'loss_history' in locals() and len(loss_history) > 0:
    plt.figure(figsize=(10, 5))
    
    # Smooth the curve using a moving average
    window = max(1, len(loss_history) // 50)
    smoothed_loss = np.convolve(loss_history, np.ones(window)/window, mode='valid')
    
    plt.plot(smoothed_loss, color='blue', linewidth=2)
    plt.title("Phase 1: Continuous Thought Training Loss (Smoothed)")
    plt.xlabel("Training Steps")
    plt.ylabel("Cross Entropy Loss")
    plt.grid(True, linestyle='--', alpha=0.7)
    plt.show()
else:
    print("Please add 'loss_history = []' and 'loss_history.append(loss.item())' to your Phase 1 training loop to use this plot.")

In [ ]:
# CELL 6: Phase 2 - Global Truth Vector Extraction
print("Starting Phase 2: Calculating Global Truth Vector...")
coconut_model.eval()

ds_phase2 = get_hf_dataset(data_phase2, tokenizer)
correct_latents = []
wrong_latents = []
n_latents_infer = configs.max_latent_stage * configs.c_thought

for sample in tqdm(ds_phase2, desc="Phase 2 Inference"):
    prompt = sample["question"] + "\n<|start-latent|>" + "<|latent|>" * n_latents_infer + "<|end-latent|>"
    input_ids = tokenizer.encode(prompt, return_tensors="pt").to(configs.device)
    ground_truth = sample["ground_truth"].replace(",", "").strip()
    
    for _ in range(configs.num_generations_per_sample):
        generated_ids, mean_latent, _ = coconut_model.generate_with_latents(
            input_ids, 
            max_new_tokens=64,
            temperature=configs.generation_temperature 
        )
        
        output_text = tokenizer.decode(generated_ids[0], skip_special_tokens=True)
        pred = output_text.split("####")[-1].strip() if "####" in output_text else output_text.split(" ")[-1].strip()
        pred_clean = pred.replace(",", "").replace(".", "").split(" ")[0]
        
        if ground_truth in pred_clean or pred_clean in ground_truth:
            correct_latents.append(mean_latent)
        else:
            wrong_latents.append(mean_latent)

print(f"Total Correct Paths: {len(correct_latents)}")
print(f"Total Incorrect Paths: {len(wrong_latents)}")

if len(correct_latents) > 0 and len(wrong_latents) > 0:
    mean_correct = torch.mean(torch.stack(correct_latents), dim=0)
    mean_wrong = torch.mean(torch.stack(wrong_latents), dim=0)
    truth_vector = (mean_correct - mean_wrong).to(configs.device)
else:
    print("WARNING: Missing either correct or wrong examples. Vector will be zero.")
    truth_vector = torch.zeros((1, model.config.hidden_size)).to(configs.device)

print(f"Computed Global Truth Vector. Shape: {truth_vector.shape}")
l2_norm = torch.norm(truth_vector).item()
print(f"Truth Vector L2 Norm (Magnitude): {l2_norm:.6f}")

if l2_norm < 1e-3:
    print("WARNING: Vector magnitude is extremely small.")

In [ ]:
# CELL 6.2: Latent Space Geometry (Plotly Fix for Global Extraction)
import torch
import plotly.graph_objects as go
import plotly.io as pio
import numpy as np

# --- PLOTLY FIX: Force Kaggle to render the graph in a robust iframe ---
pio.renderers.default = 'iframe'

print("Analyzing Latent Space Geometry using PyTorch...")

# --- DATA FIX: Use the lists directly from Global Extraction (Cell 6) ---
if 'correct_latents' in locals() and 'wrong_latents' in locals() and len(correct_latents) > 2 and len(wrong_latents) > 2:
    
    # 1. Flatten tensors to 1D arrays
    flat_correct = [vec.cpu().float().view(-1) for vec in correct_latents]
    flat_wrong = [vec.cpu().float().view(-1) for vec in wrong_latents]
    
    # Combine into a single PyTorch tensor (N, D)
    all_latents = torch.stack(flat_correct + flat_wrong)
    labels = [1] * len(flat_correct) + [0] * len(flat_wrong)
    
    # 2. Native PyTorch PCA
    all_latents_centered = all_latents - all_latents.mean(dim=0)
    U, S, V = torch.pca_lowrank(all_latents_centered, q=2)
    reduced_latents = torch.matmul(all_latents_centered, V[:, :2]).numpy()
    
    # 3. Plotly Interactive Graph
    fig = go.Figure()
    
    # Incorrect Paths (Red crosses)
    wrong_mask = np.array(labels) == 0
    fig.add_trace(go.Scatter(
        x=reduced_latents[wrong_mask, 0],
        y=reduced_latents[wrong_mask, 1],
        mode='markers',
        name='Incorrect Paths',
        marker=dict(color='red', symbol='x', opacity=0.6, size=8)
    ))
    
    # Correct Paths (Blue circles)
    correct_mask = np.array(labels) == 1
    fig.add_trace(go.Scatter(
        x=reduced_latents[correct_mask, 0],
        y=reduced_latents[correct_mask, 1],
        mode='markers',
        name='Correct Paths',
        marker=dict(color='blue', symbol='circle', opacity=0.6, size=8)
    ))
    
    fig.update_layout(
        title="Latent Space PCA: Correct vs. Incorrect Thoughts",
        xaxis_title="Principal Component 1",
        yaxis_title="Principal Component 2",
        template="plotly_white",
        width=800,
        height=600
    )
    
    fig.show()
else:
    print("Not enough latents generated to perform PCA. Please ensure Cell 6 ran successfully.")

In [ ]:
# CELL 7: Phase 4 - Inference-Time Intervention on FULL TEST SET
print("Starting Phase 4 Evaluation on Full Test Set...")
coconut_model.eval()

# --- EVALUATING ON THE FULL TEST SET ---
eval_subset = test_data 
n_latents_infer = configs.max_latent_stage * configs.c_thought

baseline_correctness = []
experiment_results = []

for alpha in configs.alpha_sweep:
    print(f"\nEvaluating Alpha = {alpha}")
    correct = 0
    total = len(eval_subset)
    flips = 0
    faithfulness_list = []
    
    for idx, sample in enumerate(tqdm(eval_subset, desc=f"Eval α={alpha}")):
        prompt = sample["question"] + "\n<|start-latent|>" + "<|latent|>" * n_latents_infer + "<|end-latent|>"
        input_ids = tokenizer.encode(prompt, return_tensors="pt").to(configs.device)
        ground_truth = sample["answer"].replace(",", "").strip()

        with torch.no_grad():
            generated_ids, _, faith_score = coconut_model.generate_with_latents(
                input_ids, 
                max_new_tokens=64,
                temperature=0.0, 
                steering_vector=truth_vector if alpha > 0 else None,
                alpha=alpha,
                gamma=configs.alpha_decay
            )

        output_text = tokenizer.decode(generated_ids[0], skip_special_tokens=True)
        pred = output_text.split("####")[-1].strip() if "####" in output_text else output_text.split(" ")[-1].strip()
        pred_clean = pred.replace(",", "").replace(".", "").split(" ")[0]

        is_correct = (ground_truth in pred_clean or pred_clean in ground_truth)
        
        if is_correct:
            correct += 1
            
        if alpha > 0:
            faithfulness_list.append(faith_score)

        if alpha == 0.0:
            baseline_correctness.append(is_correct)
        else:
            if not baseline_correctness[idx] and is_correct:
                flips += 1

    acc = correct / total
    baseline_errors = total - sum(baseline_correctness)
    flip_rate = (flips / baseline_errors) if baseline_errors > 0 else 0.0
    avg_faith = np.mean(faithfulness_list) if faithfulness_list else 0.0
    
    experiment_results.append({
        "alpha": alpha, 
        "accuracy": acc, 
        "flip_rate": flip_rate, 
        "trajectory_faithfulness": avg_faith
    })

print("\n" + "="*50)
print(f"{'Alpha':<8} | {'Accuracy':<10} | {'Flip Rate':<10} | {'Faithfulness'}")
print("-" * 50)
for res in experiment_results:
    print(f"{res['alpha']:<8} | {res['accuracy']:.2%}     | {res['flip_rate']:.2%}     | {res['trajectory_faithfulness']:.4f}")
print("="*50)

In [ ]:
# CELL 8: Deep Dive - Target Answer Confidence Shift
import torch.nn.functional as F

print("Running Single-Sample Confidence Analysis...")
coconut_model.eval()

# Pick a specific sample from the eval set
sample = eval_subset[0] 
prompt = sample["question"] + "\n<|start-latent|>" + "<|latent|>" * n_latents_infer + "<|end-latent|>"
input_ids = tokenizer.encode(prompt, return_tensors="pt").to(configs.device)
ground_truth = sample["answer"].replace(",", "").strip()

# Tokenize the ground truth to track its probability
target_token_ids = tokenizer.encode(ground_truth, add_special_tokens=False)
if len(target_token_ids) > 0:
    first_target_token = target_token_ids[0]
    target_token_string = tokenizer.decode([first_target_token])
    
    print(f"Question: {sample['question']}")
    print(f"Tracking Probability for target token: '{target_token_string}' (ID: {first_target_token})")
    print("-" * 50)
    
    for alpha in configs.alpha_sweep:
        with torch.no_grad():
            outputs = coconut_model(
                input_ids, 
                attention_mask=torch.ones_like(input_ids), 
                steering_vector=truth_vector if alpha > 0 else None,
                alpha=alpha,
                gamma=configs.alpha_decay
            )
            
            # Look at the logits for the very first generated token after the prompt
            next_token_logits = outputs.logits[:, -1, :]
            probs = F.softmax(next_token_logits, dim=-1)
            
            # Get the probability of our target token
            target_prob = probs[0, first_target_token].item()
            
            # Get the actual top predicted token
            top_token_id = torch.argmax(next_token_logits, dim=-1).item()
            top_token_prob = probs[0, top_token_id].item()
            top_token_str = tokenizer.decode([top_token_id])
            
            print(f"Alpha {alpha:>4.1f} | Target Token Prob: {target_prob:>6.2%} | Top Pred: '{top_token_str}' ({top_token_prob:.2%})")
else:
    print("Could not parse ground truth for token tracking.")